### Packages Imported

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

### Reading the file 

In [0]:
df = spark.read.format("csv")\
             .option("header", True)\
             .option("inferSchema", True)\
             .load("/Workspace/Users/edwinvictor73@gmail.com/Data-science/raw/ipl_ball_by_ball.csv")

In [0]:
df.limit(10).display()

### Cleaning the column season before filtering as the value in season column is in two different form

In [0]:
df_season_col_cleaned = df.withColumn("season", \
    when(col("season").rlike(r"^\d{4}/\d{2}$"),year(col("date")))\
        .otherwise(col("season")))\
        .withColumn("season", col("season").cast(IntegerType()))\
        .filter(col("season") >= 2020)
        
                        
    

In [0]:
df_season_col_cleaned.limit(10).display()


### Creating new columns for batsmen runs 

In [0]:
df_season_col_cleaned.limit(5).display()

In [0]:
df_total_runs = df_season_col_cleaned.groupby(col("batter"))\
    .agg(sum("batter_runs").alias("total_runs"),\
     sum(when(col("is_powerplay") == 1, col("batter_runs"))
         .otherwise(0)
         ).alias("Runs_scored_in_powerplay"),\
     sum(when(col("is_middle_overs") == 1, col("batter_runs"))
         .otherwise(0)
        ).alias("Runs_scored_in_middle_overs"),\
     sum(when(col("is_death_overs") == 1, col("batter_runs"))
         .otherwise(0)
         ).alias("Runs_scored_in_death_overs")
     )

In [0]:
df_total_runs.sort("Runs_scored_in_death_overs", ascending=False).limit(20).display()

### Batsmen balls faced 

In [0]:
df_season_col_cleaned_1 = df.withColumn("season", \
                            when(col("season").rlike(r"^\d{4}/\d{2}$"), year(col("date")))\
                                .otherwise(col("season")))\
                                .withColumn("season", col("season").cast(IntegerType()))      

In [0]:
df_season_col_filtered_1 = df_season_col_cleaned_1.filter(col("season") >= 2020)

In [0]:
df_balls_faced = (
    df_season_col_filtered_1
    .groupBy("batter")
    .agg(
        count(
            when(
                (col("wides") == 0) & (col("noballs") == 0),
                col("ball_in_over")
            )
        ).alias("total_balls_faced"),

        count(
            when(
                ((col("wides") == 0) & (col("noballs") == 0)) &
                (col("is_powerplay") == 1),
                col("ball_in_over")
            )
        ).alias("balls_faced_in_powerplay"),

        count(
            when(
                ((col("wides") == 0) & (col("noballs") == 0)) &
                 (col("is_middle_overs") == 1),
                 (col("ball_in_over"))
               )
           ).alias("balls_faced_in_middle_overs"),
        
        count(
            when(
                ((col("wides") == 0 ) & (col("noballs") == 0)) &
                 (col("is_death_overs") == 1) , 
                 (col("ball_in_over"))
                ) 
            ).alias("balls_faced_in_death_overs")
    )
)


In [0]:
df_balls_faced.sort(col("total_balls_faced"), ascending=False).limit(10).display()

In [0]:
df_season_col_filtered_1.limit(25).display()

### Dismissals in different phases of the game

In [0]:
df.limit(5).display()

In [0]:
#Cleaning season column 

df_cleaned_season_2 = df.withColumn("season", 
                                    when(
                                        col("season").rlike(r"^\d{4}/\d{2}$"), year(col("date"))
                                       ).otherwise(col("season"))
                                    ).withColumn(
                                        "season", col("season").cast(IntegerType())
                                    ).filter(col("season") >=2020)
                                    

In [0]:
df_dismissals = df_cleaned_season_2.groupby("batter")\
    .agg(
        count(
            when(
                 col("is_wicket") == 1,
                 col("is_wicket")
               )
           ).alias("total_dismissals"),
        
        count(
            when(
                 ((col("is_wicket") == 1) & (col("is_powerplay") == 1)),
                 col("is_wicket")
               )
            ).alias("total_dismissals_in_powerplay"),
        
        count(
            when(
                 ((col("is_wicket") == 1 ) & (col("is_middle_overs") == 1)),
                 col("is_wicket")
               )
            ).alias("total_dismissals_in_middle_overs"),
        count(
            when(
                ((col("is_wicket") == 1) & (col("is_death_overs") == 1)),
                 col("is_wicket") 
                )
            ).alias("total_dismissals_in_death_overs")
     )
               


In [0]:
df_dismissals.sort("total_dismissals", ascending = False).limit(10).display()

In [0]:
df_total_runs.select("batter").distinct().count()


In [0]:
df_dismissals.select("batter").distinct().count()


In [0]:
df_balls_faced.select("batter").distinct().count()

### Joining the dataframes 

In [0]:
batsman_overall_statistics_dataframe =(
    df_total_runs.join(
                        df_balls_faced, on = "batter", how ="left"
                     )
                  .join(
                      df_dismissals, on = "batter", how="left"
                  )
)

### batsmen metrics average, strikerate in different phases of game 

In [0]:
batsman_df_statistics = batsman_overall_statistics_dataframe.withColumn(
    "Batsman_average", when(
                             ((col("total_runs") > 0) & (col("total_dismissals") > 0)),\
                                 round(try_divide(col("total_runs"), col("total_dismissals")),2)
                          )
    .otherwise(col("total_runs"))
)\
.withColumn(
    "Batsman_Strike_rate", 
    when( 
         col("total_runs") > 0,
         round((try_divide(col("total_runs"), col("total_balls_faced")) * 100), 2)  
       ).otherwise(0)
)\
.withColumn("Batsman_average_in_powerplay", 
            when(
                (col("Runs_scored_in_powerplay") > 0) & (col("total_dismissals_in_powerplay") > 0), \
                    round(try_divide(col("Runs_scored_in_powerplay"), col("total_dismissals_in_powerplay")), 2)  
               ).otherwise(col("Runs_scored_in_powerplay"))
)\
.withColumn("Batsman_Strike_rate_in_powerplay",
            when(
                (col("Runs_scored_in_powerplay")  > 0) & (col("balls_faced_in_powerplay") > 0),\
                   round(try_divide(col("Runs_scored_in_powerplay"), col("balls_faced_in_powerplay")) * 100, 2)
               ).otherwise(None)
)\
.withColumn("Batsman_average_in_middle_overs",
            when(
                (col("Runs_scored_in_middle_overs") > 0) & (col("total_dismissals_in_middle_overs") > 0), \
                  round(try_divide(col("Runs_scored_in_middle_overs"), col("total_dismissals_in_middle_overs")), 2)
               ).otherwise(0)
)\
.withColumn("Batsman_Strike_rate_in_middle_overs",
            when( 
                 (col("Runs_scored_in_middle_overs") > 0) & (col("balls_faced_in_middle_overs") > 0), \
                   round(try_divide(col("Runs_scored_in_middle_overs"), col("balls_faced_in_middle_overs")) *100 ,2)
                ).otherwise(None)
)\
.withColumn("Batsman_average_in_death_overs",
            when(
                (col("Runs_scored_in_death_overs") > 0) & (col("total_dismissals_in_death_overs") > 0), \
                  round(try_divide(col("Runs_scored_in_death_overs"), col("total_dismissals_in_death_overs")), 2)
               ).otherwise(0)
)\
.withColumn("Batsman_Strike_rate_in_death_overs",
            when(
                 (col("Runs_scored_in_death_overs") > 0) & (col("balls_faced_in_death_overs") > 0), \
                   round(try_divide(col("Runs_scored_in_death_overs"), col("balls_faced_in_death_overs")) * 100, 2)
                ).otherwise(None)
)
   

In [0]:
batsman_df_statistics.select("batter", "Batsman_average", "Batsman_Strike_rate",
                             "Batsman_average_in_powerplay","Batsman_Strike_rate_in_powerplay",
                               "Batsman_average_in_middle_overs","Batsman_Strike_rate_in_middle_overs",
                               "Batsman_average_in_death_overs", "Batsman_Strike_rate_in_death_overs"
                               )\
.sort("Batsman_average_in_death_overs", ascending = False).limit(10).display()

In [0]:
batsman_df_statistics.limit(10).display()

In [0]:
batsman_df_statistics.select("batter").distinct().count()

In [0]:
df_cleaned_season_2.limit(5).display()

### Batsmen dot ball metrics is different phases of the game 

In [0]:
df_dot_ball = df_cleaned_season_2.filter((col("batter_runs") == 0) & 
                                         (col("season") >= 2020) &
                                         (col("wides") == 0) &
                                         (col("legbyes") == 0) &
                                         (col("byes") == 0) &
                                         (col("noballs") == 0) &
                                         (col("penalty") == 0) &
                                         (col("is_wicket") == 0)
                                    )

In [0]:
df_batter_dot_balls = df_dot_ball.groupBy("Batter")\
    .agg(
        count("ball_in_over").alias("Dot_balls"),
        count(
            when(
                 col("is_powerplay") == 1, col("ball_in_over")
               )
            ).alias("Dot_balls_in_powerplay"),
        count(
            when(
                col("is_middle_overs") == 1, col("ball_in_over")
            )
            ).alias("Dot_balls_in_middle_overs"),
        count(
            when(
                col("is_death_overs") == 1, col("ball_in_over")
            )
            ).alias("Dot_balls_in_death_overs")
)\

df_batter_dot_balls.sort("Dot_balls", ascending=False).limit(10).display()

In [0]:
df_batter_dot_balls.select("Batter").distinct().count()

In [0]:
df_batsman_dot_ball_df_merge = batsman_df_statistics.join(
    df_batter_dot_balls, 
    on = "batter", 
    how = "left"
)

In [0]:
df_batsman_dot_ball_df_merge.limit(3).display()

In [0]:
df_batsmen_dot_ball_metric = (
          df_batsman_dot_ball_df_merge.withColumn("Dot_ball_percentage",
                                   when(
                                        col("total_balls_faced") > 0,
                                          round(
                                                try_divide(col("Dot_balls"), col("total_balls_faced")) * 100,2
                                            )
).otherwise(0).alias("Dot_ball_percentage")                             
)\
.withColumn("Dot_ball_percentage_in_powerplay",
            when(
                  (col("Dot_balls_in_powerplay") > 0),
                  round(
                        try_divide(col("Dot_balls_in_powerplay"), col("balls_faced_in_powerplay")) * 100, 2
                  )
               ).otherwise(0).alias("Dot_ball_percentage_in_powerplay")
)\
.withColumn("Dot_ball_percentage_in_middle_overs",
            when(
                  (col("Dot_balls_in_middle_overs") > 0),
                  round(
                      try_divide(col("Dot_balls_in_middle_overs"), col("balls_faced_in_middle_overs")) * 100, 2
                  )
                  ).otherwise(0).alias("Dot_ball_percentage_in_middle_overs")
)\
.withColumn("Dot_ball_percentage_in_death_overs",
            when(
                  (col("Dot_balls_in_death_overs") > 0),
                  round(
                      try_divide(col("Dot_balls_in_death_overs"), col("balls_faced_in_death_overs")) * 100, 2
                  )
                  ).otherwise(0).alias("Dot_ball_percentage_in_death_overs")
               )
)
df_batsmen_dot_ball_metric.select("batter", "Dot_ball_percentage").sort("Dot_ball_percentage", ascending=False).limit(30).display()

In [0]:
df_batsmen_dot_ball_metric.select("batter","Dot_ball_percentage", "Dot_ball_percentage_in_powerplay", "Dot_ball_percentage_in_middle_overs", "Dot_ball_percentage_in_death_overs").display()

In [0]:
df_batsmen_statistics_friday = (
    batsman_df_statistics.alias("stats")
    .join(
        df_batsmen_dot_ball_metric.alias("dot"),
        on="batter",
        how="left"
    )
    .select(
        col("stats.batter"),
        col("stats.Batsman_average"),
        col("stats.Batsman_Strike_rate"),
        col("dot.Dot_ball_percentage"),
        col("dot.Dot_ball_percentage_in_powerplay"),
        col("dot.Dot_ball_percentage_in_middle_overs"),
        col("dot.Dot_ball_percentage_in_death_overs"),
        col("stats.Batsman_average_in_powerplay"),
        col("stats.Batsman_average_in_middle_overs"),
        col("stats.Batsman_average_in_death_overs"),
        col("stats.Batsman_Strike_rate_in_powerplay"),
        col("stats.Batsman_Strike_rate_in_middle_overs"),
        col("stats.Batsman_Strike_rate_in_death_overs")
    )
)

df_batsmen_statistics_friday.limit(3).display()

### Batsmen boundary frequency metrics across different phases of game 

In [0]:
df_boundary_metrics_cleaned =( 
                               df.withColumn("season",
                                            when(col("season").rlike(r"^\d{4}/\d{2}$"),
                                                 year(col("date"))
                                               ).otherwise(col("season"))
                                       ).withColumn("season", col("season").cast(IntegerType()))
.filter(col("season") >= 2020)
)

df_boundary_metrics_cleaned.limit(10).display()


In [0]:
df_tot_boundaries = (
df_boundary_metrics_cleaned.groupBy("batter")\
                           .agg(
                               count(
                                       when(((col("Batter_runs") == 4 ) | (col("Batter_runs") == 6)), 1)
                                    ).alias("Total_boundaries"),
                               count(
                                      when(((col("Batter_runs") == 4) | (col("Batter_runs") == 6)) & 
                                           (col("is_powerplay") == 1), 1)
                                   ).alias("Total_boundaries_in_powerplay"),
                               count(
                                      when(((col("Batter_runs") == 4) | (col("Batter_runs") == 6)) & 
                                           (col("is_middle_overs") == 1), 1)
                                   ).alias("Total_boundaries_in_middle_overs"),
                               count(
                                      when(((col("Batter_runs") == 4) | (col("Batter_runs") == 6)) & 
                                           (col("is_death_overs") == 1), 1)
                                   ).alias("Total_boundaries_in_death_overs"),
                               count(
                                   when((col("Batter_runs") == 4), 1)
                               ).alias("Total_fours"),
                               count(
                                   when((col("Batter_runs") == 6), 1)
                               ).alias("Total_sixes"),
                               count(
                                   when((col("Batter_runs") == 4) & (col("is_powerplay") == 1), 1)
                               ).alias("Total_fours_in_powerplay"),
                               count(
                                   when((col("Batter_runs") == 6) & (col("is_powerplay") == 1), 1)
                               ).alias("Total_sixes_in_powerplay"),
                               count(
                                   when((col("Batter_runs") == 4) & (col("is_middle_overs") == 1), 1)
                               ).alias("Total_fours_in_middle_overs"),
                               count(
                                   when((col("Batter_runs") == 6) & (col("is_middle_overs") == 1), 1)
                               ).alias("Total_sixes_in_middle_overs"),
                               count(
                                   when((col("Batter_runs") == 4) & (col("is_death_overs") == 1), 1)
                               ).alias("Total_fours_in_death_overs"),
                               count(
                                   when((col("Batter_runs") == 6) & (col("is_death_overs") == 1), 1)
                               ).alias("Total_sixes_in_death_overs")
)
                           
)
                               
                
                               
                               

In [0]:
df_tot_boundaries.limit(10).display()

In [0]:
df_batsmen_statistics_friday.limit(1).display()

In [0]:
df_boundary_stats = df_batsmen_statistics_friday.join(
                 df_tot_boundaries,
                 on = "batter",
                 how = "left")

In [0]:
df_batsman_dot_ball_df_merge.limit(1).display()

In [0]:
df_boundaries_ball_df = (
df_tot_boundaries.alias("Boundaries").join(
    df_batsman_dot_ball_df_merge.alias("Ball_faced"),
    on = "batter",
    how = "left"
).select(col("Boundaries.*"),
         col("Ball_faced.total_balls_faced"),
         col("Ball_faced.balls_faced_in_powerplay"),
         col("Ball_faced.balls_faced_in_middle_overs"),
         col("Ball_faced.balls_faced_in_death_overs")
         )
)

In [0]:
df_boundaries_ball_df.limit(1).display()

In [0]:
df_boundaries_metrics = df_boundaries_ball_df.withColumn(
    "Balls_per_Boundary", round(
                                 try_divide(col("total_balls_faced"), 
                                            col("Total_boundaries")
                                            ), 2
                                )
).withColumn("Balls_per_Boundary_in_powerplay",
              round(try_divide(col("balls_faced_in_powerplay"), 
                               col("Total_boundaries_in_powerplay")
                               ), 2
                    )
).withColumn("Balls_per_Boundary_in_middle_overs", 
             round(try_divide(col("balls_faced_in_middle_overs"), 
                              col("Total_boundaries_in_middle_overs")
                              ), 2
                   )
).withColumn("Balls_per_Boundary_in_death_overs", 
             round(try_divide(col("balls_faced_in_death_overs"), 
                              col("Total_boundaries_in_death_overs")
                              ), 2
                   )
).withColumn("4s_per_Ball", round(
                                 try_divide(
                                     col("total_balls_faced"),
                                     col("Total_fours") 
                                        ), 2
                                )
).withColumn("4s_per_Ball_in_powerplay",
              round(try_divide(col("balls_faced_in_powerplay"),
                               col("Total_fours_in_powerplay")
                               ), 2
                    )
).withColumn("4s_per_Ball_in_middle_overs", 
             round(try_divide(
                 col("balls_faced_in_middle_overs"),
                 col("Total_fours_in_middle_overs") 
                              ), 2
                   )
).withColumn("4s_per_Ball_in_death_overs", 
             round(try_divide(
                 col("balls_faced_in_death_overs"),
                 col("Total_fours_in_death_overs")
                              ), 2
                   )
).withColumn("6s_per_Ball", round(
                                 try_divide(
                                     col("total_balls_faced"),
                                     col("Total_sixes")
                                            ), 2
                                )
).withColumn("6s_per_Ball_in_powerplay",
              round(try_divide(
                  col("balls_faced_in_powerplay"),
                  col("Total_sixes_in_powerplay")
                               ), 2
                    )
).withColumn("6s_per_Ball_in_middle_overs", 
             round(try_divide(
                 col("balls_faced_in_middle_overs"),
                 col("Total_sixes_in_middle_overs")

                              ), 2
                   )
).withColumn("6s_per_Ball_in_death_overs", 
             round(try_divide(
                 col("balls_faced_in_death_overs"),
                 col("Total_sixes_in_death_overs")   
                              ), 2
                   )
)

In [0]:
df_boundaries_metrics.sort("Balls_per_Boundary", ascending = True).filter(col("Balls_per_Boundary_in_powerplay") > 0).select("batter","Balls_per_Boundary","Balls_per_Boundary_in_middle_overs", "Balls_per_Boundary_in_death_overs").limit(10).display()

### Did investigation about dataset it contains data until may 1 

In [0]:
df_balls_faced.filter(col("batter").like("%Allen%")).display()

In [0]:
df.filter((year(col("date")) >= 2026)).sort("date", ascending=False).display()

### Using window functions to get runs scored in each innings 


In [0]:
innings_window = Window.partitionBy("match_id", "Batter")
 

In [0]:
df_batsman_runs_each_innings = df_cleaned_season_2.withColumn("Runs_Scored", 
                                                             sum("Batter_Runs").over(innings_window)
                                                        )\
.select("batter", col("bowling_team").alias("Vs"),"Runs_Scored").distinct()\
.sort("batter", ascending = True)

In [0]:
df_runs_bucket = df_batsman_runs_each_innings.withColumn(
    "Runs_Bucket",
    when(col("Runs_Scored") < 15, "0-15")
    .when(col("Runs_Scored") < 30, "15-30")
    .when(col("Runs_Scored") < 50, "30-50")
    .otherwise("50+")
)

In [0]:
df_runs_bucket_pivoted = (
    df_runs_bucket
    .groupBy("Batter")
    .pivot("Runs_Bucket")
    .agg(count("Runs_Bucket").alias("Times"))
)

In [0]:
df_runs_bucket_pivoted.sort("30-50", ascending=False).display()

In [0]:
df_boundaries_metrics.display()

In [0]:
df_batsman_boundary_runs_bucket_metrics =(
df_boundaries_metrics.join(df_runs_bucket_pivoted,
                           on = "batter",
                           how = "left")
)

In [0]:
df_batsman_boundary_runs_bucket_metrics.distinct().count()

In [0]:
df_batsmen_statistics_tuesday = (
df_batsmen_statistics_friday.join(
    df_batsman_boundary_runs_bucket_metrics,
                                  on = "batter",
                                  how = "left"
                                  )
)


In [0]:
df_batsmen_statistics_tuesday.filter(col("6s_per_Ball_in_death_overs").isNotNull()).sort("6s_per_Ball_in_death_overs", ascending=True).limit(10).display()

In [0]:
df_batsmen_statistics_tuesday.columns

### Writing as Managed Delta table 

In [0]:
df_batsmen_statistics_tuesday.write\
                        .format("delta")\
                        .mode("overwrite")\
                        .saveAsTable("Batsmen_Statistics")

In [0]:
spark.catalog.currentDatabase()